# Solar filament segmentation: the whole pipeline in one notebook

This notebook reproduces our final submission to the Kaggle competition
*filament-segmentation-2026* (Solar Filament Segmentation Challenge 2026), from the 180 test
images to the CSV file `v2_blend_r2_exp8_coat1_pq0.16.csv`. The submission blends three UNet
families, and a LightGBM scorer then decides which pieces of the blended map are filaments.
Its out-of-fold PQ is 0.4568, and its public leaderboard score is 0.39.

The pipeline in one line: **image → three input planes → UNets (5 folds × 4 flips each) →
blended probability map → connected pieces (candidates) → a scorer keeps the pieces whose
`p·q` clears a cut → one RLE row per filament.**

Each stage calls the repository's own scripts through their `main(argv)`:
`scripts/cache_probs.py`, `blend_caches.py` and `submission_from_cache.py`. The notebook
therefore runs the same code that wrote the submitted file. The short helpers behind the
figures and the checks are in `scripts/pipeline_nb.py`.

**What you need, and nothing else:** this repository with its pinned `requirements.txt`, and
the competition's test images. The trained weights are in the repository, under `weights/`:
15 checkpoints (3 families × 5 folds) and one scorer (10 small text files and a feature list).
No model is retrained. GitHub refuses files above 100 MB, so each checkpoint is stored as two
parts; the setup cell joins them once and checks every file against `weights/SHA256SUMS`.

**Two ways to run.**
- `FROM_MAPS = False` (the default) computes the probability maps from the checkpoints: 60
  forward passes per test image (3 families × 5 folds × 4 flips), so in practice a GPU.
- `FROM_MAPS = True` skips section 4 and starts from test maps already in `WORK_DIR/cache`. The
  rest runs on a CPU in a few minutes.

The last section compares the file written here with the submitted file, which is kept in
`submissions/`.

**Without Jupyter:** `PYTHONPATH=src python scripts/run_notebook.py notebooks/pipeline.ipynb
--set LIMIT=3` runs the code cells in order in one Python process. Every variable of the
configuration cell can be set the same way with `--set NAME=VALUE`.


## 1. Setup

The first cell finds the repository root and puts its package (`src/filseg`) and scripts
(`scripts/`) on the import path. The second cell holds the whole configuration. It is the
only cell to edit, and it is tagged `parameters`, so `run_notebook.py --set` (or papermill)
can override it.

In [ ]:
import os
import sys
import time
from pathlib import Path

import torch

# The notebook sits in <repo>/notebooks; the pipeline code is the repository's package
# (src/filseg) and its command-line scripts (scripts/), imported from the checkout.
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "filseg").is_dir()), None)
if REPO is None:
    raise RuntimeError("run this notebook from inside the repository (its notebooks/ folder)")
for sub in ("src", "scripts"):
    if str(REPO / sub) not in sys.path:
        sys.path.insert(0, str(REPO / sub))
print("repository:", REPO)

In [ ]:
# Configuration: the only cell to edit (run_notebook.py --set NAME=VALUE overrides it).
DATA_ROOT = os.environ.get("FILSEG_DATA_ROOT",          # the MAGFiLO_1.0_Kaggle_2026 folder
                           str(REPO / "filament-segmentation-2026" / "MAGFiLO_1.0_Kaggle_2026"))
WEIGHTS_DIR = str(REPO / "weights")                 # <family>_fold<k>.pt, stored as parts: 3 families x 5 folds
SCORER_DIR = str(REPO / "weights" / "scorer")       # <version>_{p,q}_fold<k>.txt, <version>.features.json
WORK_DIR = str(REPO / "pipeline_out")               # test maps (cache/) and CSVs (submissions/) go here
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LIMIT = None             # None = all 180 test images; an int N = the first N (a quick run, PARTIAL_ CSVs)
FROM_MAPS = False        # True = skip inference and reuse the test maps already in WORK_DIR/cache
REFERENCE_DIR = str(REPO / "submissions")   # the submitted CSV, compared in section 6 (None = skip)
EXAMPLE_STEM = None      # the test image the figures show; None = the first one processed

In [ ]:
DATA_ROOT, WEIGHTS_DIR, SCORER_DIR, WORK_DIR = map(Path, (DATA_ROOT, WEIGHTS_DIR, SCORER_DIR, WORK_DIR))
os.environ["FILSEG_DATA_ROOT"] = str(DATA_ROOT)     # filseg.paths reads the images from here
CACHE_DIR, SUB_DIR = WORK_DIR / "cache", WORK_DIR / "submissions"
SUB_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import pandas as pd

import blend_caches
import cache_probs
import pipeline_nb as nb
import submission_from_cache
from filseg import data as fdata
from filseg import weights
from filseg.config import load_config
from filseg.dataset import model_input
from filseg.model import build_from_config

if not FROM_MAPS:
    joined = weights.assemble(WEIGHTS_DIR)        # join the checkpoint parts once, checked against SHA256SUMS
    print(f"{len(joined)} checkpoints ready in {WEIGHTS_DIR}")

TEST_STEMS = fdata.list_stems("test")[:LIMIT]       # sorted, as every script lists them
STEM = EXAMPLE_STEM or TEST_STEMS[0]
assert STEM in TEST_STEMS, f"EXAMPLE_STEM {STEM} is not among the {len(TEST_STEMS)} images of this run"
print(f"{len(TEST_STEMS)} test images | device {DEVICE} | from maps: {FROM_MAPS} | work folder {WORK_DIR}")

## 2. Data and preprocessing

Each test image is a 2048 × 2048 8-bit grayscale GONG H-alpha frame. Filaments are thin dark
ribbons on the solar disc. The median filament is only about 7% darker than its surroundings,
while limb darkening dims the edge of the disc by about 19%. The network therefore does not
see the raw frame alone. Two steps run before any model, at inference exactly as in training:

1. **Disc fit** (`filseg.data.detect_disk`). The centre and radius are the percentile
   extents of the largest bright connected component, with its holes filled. A plain area
   threshold would overestimate the radius by about 12%.
2. **Three input planes** (`filseg.dataset.build_input`, selected by `model_input`, the call
   `cache_probs.py` makes). The planes are the flat-field ratio, the raw frame normalised
   by its on-disc level, and the normalised radius r/R. The flat-field ratio is the frame
   divided by its own smoothed illumination, which makes the faint filament contrast
   visible. The normalised radius hands the model the disc geometry for free. All three are
   resized by area averaging to the model's working size (1536 px here).

In [ ]:
image = fdata.load_image(STEM, "test")
disk = fdata.detect_disk(image)
cfg = load_config(REPO / "configs" / "scr_r2_base_1536.yaml")
x = model_input(image, disk, cfg["size"], cfg.get("inputs"))
print(f"{STEM}: {image.shape} {image.dtype} | disc centre ({disk.cx:.1f}, {disk.cy:.1f}), "
      f"radius {disk.radius:.1f} px | model input {x.shape} {x.dtype}")
nb.show_planes(image, disk, x)
plt.show()

## 3. Models

Each family is a UNet (segmentation-models-pytorch) on a timm encoder pretrained on ImageNet,
with one output channel, the filament probability. Each family is trained five times, once
per cross-validation fold; the folds are grouped by exposure, because one image carries up to
three annotators' passes. Training follows a two-stage curriculum: stage 1 at 1024 px, then a
12-epoch fine-tune at a higher resolution, warm-started from stage 1. The loss is binary
cross-entropy masked to the disc plus 0.25 × soft Dice. Each of the 1,154 annotation passes
is its own training sample. Augmentation uses flips, rotation up to ±15° and photometric
jitter; recipe r1/r2 adds zoom (0.8 to 1.25) and cutout.

| Family (config) | Encoder | Training | Runs at |
|---|---|---|---|
| `scr_r2_base_1536` | ConvNeXt-V2-tiny | 1024 px × 60 ep, then 1536 px × 12 ep (recipe r2) | 1536 |
| `exp8b_convnextv2_1536` | ConvNeXt-V2-tiny | 1024 px × 26 ep, then 1536 px × 12 ep (base augmentation) | 1536 |
| `scr_r2_coatnet1_1280` | CoAtNet-rmlp-1 | 1024 px × 60 ep, then 1280 px × 12 ep (1536 does not fit in 48 GB) | 1280 |

The configs are in `configs/`. Each one names a `base:` config it inherits from;
`filseg.config.load_config` resolves the chain. The cell below builds each family's network
from its config. When the checkpoints are in `WEIGHTS_DIR`, it loads fold 0 with the loader
the inference uses (`cache_probs.load_model`).

In [ ]:
MEMBERS = ["scr_r2_base_1536", "exp8b_convnextv2_1536", "scr_r2_coatnet1_1280"]   # the submission's blend, in order
FAMILIES = MEMBERS

rows = []
for family in FAMILIES:
    cfg = load_config(REPO / "configs" / f"{family}.yaml")
    weights = WEIGHTS_DIR / f"{family}_fold0.pt"
    net = (cache_probs.load_model(cfg, str(weights), "cpu") if weights.is_file()
           else build_from_config(cfg, pretrained=False))
    rows.append({"family": family, "encoder": cfg["encoder"], "runs at": cfg["size"],
                 "stored at": cfg["eval"].get("cache_size") or cfg["size"], "flip TTA": cfg["eval"]["tta"],
                 "parameters (M)": round(sum(p.numel() for p in net.parameters()) / 1e6, 1),
                 "fold-0 weights": "loaded" if weights.is_file() else "not in WEIGHTS_DIR"})
    del net
print(pd.DataFrame(rows).to_string(index=False))

## 4. Inference: fold-ensemble test maps

Every test image goes through all five fold models of a family, each under four flips
(none, vertical, horizontal, both; each prediction is flipped back). The 20 sigmoid maps are
averaged. Averaging probabilities before any threshold keeps the agreement between models,
which a thin filament depends on.

Each family runs at its own size. The map is then resized by area averaging to 1536 px and
stored as an 8-bit PNG (level = round(255 · p)), with a manifest that records the disc each
map was made with. Storing every family at one size lets the maps of a model run at 1280 px
be averaged with the 1536 px ones. Quantisation moves a probability by at most 1/510,
which matters only for pixels that close to a threshold.

`cache_probs.py` is the script that made the maps the submissions were cut from. Here it runs
with `--no-oof`, because the out-of-fold maps of the training images serve model selection,
not the submission. With `FROM_MAPS = True` this section only checks that the maps are there.

The figure shows the example image's three maps on the disc only. The loss ignores off-disc
pixels, so the network's output there means nothing, and candidates are only cut on the disc.

In [ ]:
if not FROM_MAPS:
    checkpoints = {f: nb.find_checkpoints(WEIGHTS_DIR, f) for f in FAMILIES}   # fail before any work
    for family, ckpts in checkpoints.items():
        t0 = time.time()
        cache_probs.main(nb.cache_argv(REPO / "configs" / f"{family}.yaml", ckpts, CACHE_DIR / family,
                                       REPO / "data" / "folds.csv", DEVICE, LIMIT))
        print(f"{family}: {len(TEST_STEMS)} test maps in {time.time() - t0:.0f}s")

missing = nb.missing_maps(CACHE_DIR, FAMILIES, TEST_STEMS)
assert not missing, f"{len(missing)} test maps missing under {CACHE_DIR}, e.g. {missing[:3]}"
print(f"test maps present for {len(TEST_STEMS)} images x {len(FAMILIES)} families under {CACHE_DIR}")

In [ ]:
nb.show_maps(CACHE_DIR, FAMILIES, STEM)
plt.show()

## 5. The submission: blend, candidates, scorer, decision

**Blend.** The maps of the submission's three families are averaged pixel by pixel with equal weights
(`blend_caches.py`). Fitted weights were tried and did not beat equal ones out of fold.

**Candidates.** The blended map is upsampled to 2048 px and thresholded at 0.40. Its
8-connected components on the disc with at least 50 px are the candidates. Pieces that
belong to one region above 0.20 are joined into one candidate. This bridge matters because
Panoptic Quality (PQ) punishes a filament split in two twice: one false positive, and a worse
IoU for the half that matches.

**Scorer.** Each candidate gets 36 features: probability statistics, shape, flat-field
contrast, position on the disc and crowding in the frame. Two LightGBM models, trained out of
fold on the training images' candidates, give two numbers: `p`, the probability that an
annotator drew the candidate, and `q`, its expected IoU if so. The five fold models of each
are averaged. The scorer asks one question: *is this candidate worth more as a likely match
than it costs as a possible false positive?* PQ credits a match with its IoU and charges half
a point for a false positive, so keeping a candidate pays roughly when `p·q` exceeds a
constant `c`. The value `c = 0.16` is the out-of-fold optimum for this blend.

**Decision.** Candidates with `p·q ≥ 0.16` are kept. An image where nothing passes keeps its
single best candidate, because every test image needs a row. Kept candidates are disjoint
connected components, so masks never overlap. Each becomes one row: the id `{image}_{n}`
and the mask as a pycocotools RLE string.

The figure below shows the image where the scorer turned down the most candidates (or
`EXAMPLE_STEM`): the blended map, every candidate with its `p·q` (kept in green, rejected in
dashed red) and the instances written to the CSV.

*Caveat:* `c` and the cut were tuned out of fold, so they are honest on the training
distribution. Even so, the public leaderboard (0.39) sits about 0.06 below the out-of-fold
PQ, and the gap has been the same for every submission we made.

In [ ]:
t0 = time.time()
blend_caches.main(nb.blend_argv([CACHE_DIR / m for m in MEMBERS], CACHE_DIR / "blend_r2_exp8_coat1"))
print(f"blend written in {time.time() - t0:.0f}s")

In [ ]:
SUBMISSION = nb.Rule(out="v2_blend_r2_exp8_coat1_pq0.16.csv", cache="blend_r2_exp8_coat1",
                     threshold=0.40, min_area=50, candidate_bridge=0.20,
                     scorer_version="scorer_v2_blend_r2_exp8_coat1", accept_pq=0.16)
submission_from_cache.main(SUBMISSION.argv(CACHE_DIR, SCORER_DIR, SUB_DIR, LIMIT))
submission_csv = SUBMISSION.written(SUB_DIR, LIMIT)

In [ ]:
shown = EXAMPLE_STEM or nb.most_rejected(SUBMISSION.scores(SUB_DIR)) or STEM   # where the scorer said no most
view = nb.frame_view(CACHE_DIR / SUBMISSION.cache, shown, SUBMISSION, SUBMISSION.scores(SUB_DIR), submission_csv)
print(view.scores.round(3).to_string(index=False))
nb.show_decision(view, SUBMISSION.threshold)
plt.show()

## 6. Checks

The file written must meet the organisers' rules, which are checked here:

- every test image has at least one row;
- every id is unique;
- no row names an image outside the test set;
- no mask is empty;
- no pixel belongs to two instances of one image (overlapping masks are rejected).

With `REFERENCE_DIR` set, the submitted file found there is compared with ours in three ways: byte for
byte, row by row, and instance by instance. An RLE string is a canonical encoding of its mask,
so equal strings mean equal pixels. On a limited run (`LIMIT`), only those images are compared,
and byte identity is not defined.

The submission must be reproduced exactly. From the cached maps (`FROM_MAPS = True`) the cell
asserts it. From maps recomputed on another GPU, floating-point noise can move a probability
by one level (1/255) at a pixel next to a threshold, so a difference is reported rather than
asserted.

In [ ]:
outputs = {"submission": submission_csv}
checks = {k: nb.check_submission(p, TEST_STEMS) for k, p in outputs.items()}
print(pd.DataFrame([c.summary() for c in checks.values()], index=list(checks)).to_string())
assert all(c.ok for c in checks.values()), "the submission file failed its checks"


In [ ]:
if REFERENCE_DIR:
    names = {"submission": SUBMISSION.out}
    found = {k: Path(REFERENCE_DIR) / n for k, n in names.items() if (Path(REFERENCE_DIR) / n).is_file()}
    assert "submission" in found, f"{SUBMISSION.out} is not in REFERENCE_DIR ({REFERENCE_DIR})"
    comparisons = {k: nb.compare_submissions(outputs[k], ref, stems=TEST_STEMS if LIMIT else None)
                   for k, ref in found.items()}
    print(pd.DataFrame([c.summary() for c in comparisons.values()], index=list(comparisons)).to_string())
    for k, c in comparisons.items():
        if c.differing:
            print(f"{k}: differs on {len(c.differing)} image(s), e.g. {c.differing[:5]}")
    got = comparisons["submission"]
    exact = got.rows_identical if LIMIT else got.byte_identical
    print(f"\nthe submission is {'' if exact else 'NOT '}reproduced exactly "
          f"({f'row for row on {got.images} images' if LIMIT else 'byte for byte'})")
    if FROM_MAPS:
        assert exact, "from the cached maps the submission must be reproduced exactly"
else:
    print("REFERENCE_DIR is not set: no comparison with the submitted file")

## Output

The file to upload is printed below. It sits in `WORK_DIR/submissions`, next to its
per-candidate score table (`*.scores.csv`). On a limited run
(`LIMIT`), every CSV carries a `PARTIAL_limit<N>_` prefix, so it cannot be mistaken for a
submission.

In [ ]:
print("submission:", submission_csv)